# DeepSpeed ZeRO Advanced Configuration

## Overview

Advanced DeepSpeed configuration for optimal training performance.

### Topics Covered
- ZeRO stage selection
- Offloading strategies
- Communication optimization
- Integration with HuggingFace

## 1. ZeRO Stage Comparison

| Stage | Shards | Memory Reduction | Communication |
|-------|--------|------------------|---------------|
| 0 | None | 1x | AllReduce |
| 1 | Optimizer | ~4x | AllReduce |
| 2 | + Gradients | ~8x | ReduceScatter + AllGather |
| 3 | + Parameters | ~Nx | AllGather (fwd+bwd) |

In [ ]:
import json

def create_zero_config(stage, offload_optimizer=False, offload_param=False):
    """Create DeepSpeed ZeRO configuration."""
    
    config = {
        "zero_optimization": {
            "stage": stage,
            "allgather_partitions": True,
            "allgather_bucket_size": 5e8,
            "reduce_scatter": True,
            "reduce_bucket_size": 5e8,
            "overlap_comm": True,
            "contiguous_gradients": True,
        },
        "bf16": {"enabled": True},
        "gradient_clipping": 1.0,
    }
    
    if stage >= 3:
        config["zero_optimization"].update({
            "stage3_prefetch_bucket_size": 5e8,
            "stage3_param_persistence_threshold": 1e6,
            "stage3_gather_16bit_weights_on_model_save": True,
        })
    
    if offload_optimizer:
        config["zero_optimization"]["offload_optimizer"] = {
            "device": "cpu",
            "pin_memory": True,
        }
    
    if offload_param:
        config["zero_optimization"]["offload_param"] = {
            "device": "cpu",
            "pin_memory": True,
        }
    
    return config

# Example configurations
print("ZeRO-2 Config:")
print(json.dumps(create_zero_config(2), indent=2))

## 2. HuggingFace Integration

In [ ]:
# HuggingFace Trainer with DeepSpeed
hf_deepspeed_config = {
    "train_batch_size": "auto",
    "train_micro_batch_size_per_gpu": "auto",
    "gradient_accumulation_steps": "auto",
    "gradient_clipping": "auto",
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {"device": "cpu"},
        "offload_param": {"device": "cpu"},
    },
    "bf16": {"enabled": "auto"},
    "zero_allow_untested_optimizer": True,
}

print("HuggingFace DeepSpeed Config:")
print(json.dumps(hf_deepspeed_config, indent=2))

## 3. Summary

### Selection Guide

```
Model fits in GPU? → ZeRO-0 (DDP)
OOM with optimizer? → ZeRO-1
OOM with gradients? → ZeRO-2
OOM with params? → ZeRO-3
Still OOM? → ZeRO-3 + Offload
```